# 트리 기반 회귀 — `DecisionTreeRegressor` · 결정 트리의 불안정성

수업 중 작성한 필기 `결정트리회귀실습01` 을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `머신러닝` p.83~90 — 12. 지도 학습을 이용한 회귀(2))

- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.
- 그림 · **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 미리 만들어 두었고, 아래 코드를 실행하면 같은 그림이 다시 그려집니다.
- 이어지는 앙상블 · 랜덤 포레스트는 [`random_forest.ipynb`](./random_forest.ipynb) 에 정리했습니다.

> 실습 데이터 `house_price.csv` (보스턴 주택 가격, 506행) 는 이 노트북과 **같은 폴더**에 있어야 합니다.

In [ ]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'     # 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'       # macOS
plt.rcParams['axes.unicode_minus'] = False            # 음수 기호(-) 깨짐 방지

## 1. 결정 트리 회귀의 동작 원리 (교재 p.86~87)

분류 트리와 **똑같이 나누고**, 리프에서 내놓는 값만 다르다.

| | 분류 트리 (`DecisionTreeClassifier`) | 회귀 트리 (`DecisionTreeRegressor`) |
|---|---|---|
| 예측 | 리프에서 **가장 많은 클래스** | 리프에 속한 학습 데이터의 **평균값** |
| 분할 기준 | 불순도(지니 · 엔트로피)가 가장 작아지게 | **평균제곱오차(MSE)** 가 가장 작아지게 |
| 노드 정보 | `gini` · `value = [클래스별 개수]` | `squared_error` · `value = 평균값` |
| 평가 | 정확도 `accuracy_score` | `root_mean_squared_error` |

리프마다 **평균값 하나**만 내놓기 때문에 예측 그래프가 **계단 모양**이 된다.

![회귀 트리가 자라는 과정](./images/anim_01_tree_regression_growing.gif)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# 교재 p.86 의 데이터 10개
X_toy = np.array([1, 3, 4, 7, 9, 10, 11, 13, 14, 16]).reshape(-1, 1)   # 특성은 2차원으로
y_toy = np.array([3, 4, 3, 15, 17, 15, 18, 7, 3, 4])

# 깊이 1 : 딱 한 번만 나누기
tree1 = DecisionTreeRegressor(max_depth=1, random_state=0).fit(X_toy, y_toy)

t = tree1.tree_
print(f"분할 조건 : X <= {t.threshold[0]}")
print(f"루트   : squared_error = {t.impurity[0]:.2f}, samples = {t.n_node_samples[0]}, value = {t.value[0][0][0]:.1f}")
print(f"왼쪽   : squared_error = {t.impurity[1]:.3f}, samples = {t.n_node_samples[1]}, value = {t.value[1][0][0]:.3f}")
print(f"오른쪽 : squared_error = {t.impurity[2]:.3f}, samples = {t.n_node_samples[2]}, value = {t.value[2][0][0]:.3f}")

> 교재 p.86 그림과 **똑같다** — `X ≤ 5.5`, 루트 `squared_error = 37.89` · `value = 8.9`, 왼쪽 `value = 3.333`, 오른쪽 `value = 11.286`.
> - `value` = 그 노드에 속한 y 의 **평균** (왼쪽 : (3 + 4 + 3) / 3 = 3.333)
> - `squared_error` = 그 노드의 **MSE** (값들이 평균에서 얼마나 퍼져 있나)

In [ ]:
# 직접 확인 : 왼쪽 리프의 평균과 MSE
left = y_toy[X_toy.ravel() <= 5.5]
print("왼쪽 리프 y :", left)
print(f"평균 = {left.mean():.3f},  MSE = {np.mean((left - left.mean()) ** 2):.3f}")

In [ ]:
# 깊이 1 과 3 의 예측선 비교 → 깊어질수록 계단이 잘게 쪼개진다
grid = np.linspace(0, 17, 500).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, depth in zip(axes, [1, 3]):
    model = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_toy, y_toy)
    ax.plot(grid, model.predict(grid), color='tab:orange', label='예측값 (리프 평균)')
    ax.scatter(X_toy, y_toy, color='tab:blue', zorder=3, label='학습 데이터')
    ax.set_title(f'max_depth = {depth}  (리프 {model.get_n_leaves()}개)')
    ax.set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].legend()
plt.show()

### 🎬 직접 애니메이션으로 보기 — 트리가 한 단계씩 깊어지는 과정

`FuncAnimation` 은 프레임마다 `draw(i)` 함수를 불러 그림을 다시 그린다. `to_jshtml()` 로 바꾸면 노트북 안에서 **재생 · 정지 · 한 프레임씩 넘기기** 버튼이 생긴다.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(8, 4))

def draw(i):
    depth = i + 1                                     # 프레임 0 → 깊이 1, 프레임 1 → 깊이 2 ...
    model = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_toy, y_toy)
    ax.clear()
    ax.plot(grid, model.predict(grid), color='tab:orange', label='예측값 (리프 평균)')
    ax.scatter(X_toy, y_toy, color='tab:blue', zorder=3, label='학습 데이터')
    mse = np.mean((y_toy - model.predict(X_toy)) ** 2)
    ax.set_title(f'max_depth = {depth} → 리프 {model.get_n_leaves()}개, 학습용 MSE {mse:.2f}')
    ax.set_ylim(0, 20.5)
    ax.legend(loc='upper left')

anim = FuncAnimation(fig, draw, frames=5, interval=1000)   # 5 프레임, 1초 간격
plt.close(fig)                                             # 정지 그림이 한 번 더 나오지 않게
HTML(anim.to_jshtml())                                     # 노트북 안에서 재생

# GIF 파일로 저장하려면 :
# anim.save('tree_growing.gif', writer='pillow', fps=1)

## 2. 보스턴 집값 예측 — `DecisionTreeRegressor` (교재 p.88~89)

```python
from sklearn.tree import DecisionTreeRegressor
dt = DecisionTreeRegressor(random_state=0)          # 모델 생성
dt.fit(X_train, y_train)                            # 학습
pred_test = dt.predict(X_test)                      # 예측

from sklearn.metrics import root_mean_squared_error as rmse
rmse_dt = rmse(y_test, pred_test)                   # 평가
```

> 트리는 한 특성 안에서 "몇 이하인가" 로만 자르므로 **표준화가 필요 없다.**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 불러오기 (노트북과 같은 폴더)
df = pd.read_csv('./house_price.csv')

# 특성(13개) / 정답(medv : 집값, 천 달러)
X_data = df.drop(columns=['medv'])
y_data = df['medv']

# 학습용 75 : 평가용 25 (Day 021 선형 회귀와 같은 분할 → 결과 비교 가능)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.25, random_state=0
)

print("학습용 :", X_train.shape, "/ 평가용 :", X_test.shape)
print("X · y 짝이 맞는가? :", (X_train.index == y_train.index).all())

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# random_state : 같은 성능의 분할 후보 중 무엇을 고를지 → 고정하면 매번 같은 트리
dt = DecisionTreeRegressor(random_state=0)
dt.fit(X_train, y_train)

print("학습 완료! 깊이 :", dt.get_depth(), "/ 리프 노드 수 :", dt.get_n_leaves())

### 트리 시각화 — 위쪽 2단계만

깊이 18 짜리 트리는 다 그리면 알아볼 수 없다. `plot_tree(max_depth=2)` 로 위쪽만 본다.

![보스턴 회귀 트리](./images/02_boston_tree_depth2.png)

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(14, 6))      # 기본 크기(6.4 x 4.8 인치)로는 글자가 겹친다
plot_tree(
    decision_tree=dt,
    feature_names=X_train.columns,
    max_depth=2,                 # 위쪽 2단계만 표시 (학습된 트리 자체는 그대로)
    filled=True,                 # 값이 클수록 진한 색
    rounded=True,
    precision=2
)
plt.show()

> **트리 읽기**
> - 루트 `lstat <= 8.13` — 저소득층 비율이 낮은 지역(135곳, 평균 31.16)과 높은 지역(244곳, 평균 17.88)으로 먼저 나눈다
> - 왼쪽은 다시 `rm <= 7.43` (방 개수) 로 → 방이 7.43 개 넘는 23곳은 평균 **44.71**
> - Day 021 상관계수에서 집값과 가장 강하게 움직였던 **`lstat`(−0.74) · `rm`(+0.70)** 이 맨 위 질문으로 나왔다

In [ ]:
from sklearn.metrics import root_mean_squared_error as rmse

# 평가용 데이터로 예측
pred_test = dt.predict(X_test)

# 정답 · 예측 나란히 보기
compare = pd.DataFrame({'정답': y_test.values, '예측': pred_test}, index=y_test.index)
print(compare.head())

print("-" * 40)
rmse_dt = rmse(y_test, pred_test)
print(f"학습용 RMSE : {rmse(y_train, dt.predict(X_train)):.2f}")
print(f"평가용 RMSE : {rmse_dt:.2f}   (Day 021 선형 회귀 5.46)")

> **결과** — 평가용 RMSE **5.41**. 선형 회귀(5.46)와 비슷하다.
> 그런데 학습용 RMSE 는 **0.00** — 끝까지 자란 트리는 리프마다 집 1~2채만 남아서 학습용 집값을 **전부 외웠다.** 과대적합이다.

> **📝 고친 부분** (`결정트리회귀실습01`)
> - 변수 이름 `y_tset` → `y_test`, 별칭 `rmes` → `rmse` (오타가 일관돼서 실행은 됐지만 읽는 사람이 헷갈린다)
> - 출력 문구 `"평균 성능:"` 인데 저장된 출력은 `결과: 5.406…` — 코드와 출력이 달랐다 → 학습용 / 평가용 RMSE 를 나란히 출력
> - `max_depth = dt.get_depth()` → `max_depth` 는 **설정값(매개변수)** 이름이라 헷갈린다. 학습된 깊이 확인은 `dt.get_depth()` 를 바로 출력
> - 주석 "트리 회귀 : 표준화, **데이터 전처리** 할 필요가 없다" → **표준화**는 필요 없지만, 결측치 · 문자열 인코딩 같은 **전처리는 여전히 필요**하다 (이 데이터는 결측치가 없을 뿐)
> - `print("X_data 출력:", X_train.index)` → `X_train` 의 **행 번호**다. 짝이 맞는지 `==` 로 한 번에 확인
> - 제목 "**학**모델 학습 시 생성된 의사 결정 **의사 결정** 트리" → 중복 정리
> - 경로 `C:\Users\Jinho\work\house_price.csv` → `./house_price.csv`
> - `plot_tree` 에 `filled` · `precision` 추가, 트리 해석 추가

## 3. 과대적합 방지 — 규제 매개변수 (교재 p.83~84)

단일 결정 트리의 장단점

| 장점 | 단점 |
|------|------|
| **최고의 설명력** — 판단 근거를 그림으로 보여 주는 '화이트박스' 모델 | **과대적합** — 깊어질수록 학습용은 100% 맞히지만 새 데이터에 약하다 |
| **비선형성** — 복잡한 비선형 관계도 직관적으로 학습 | **성능 한계** — 단일 모델로는 높은 정확도를 얻기 어렵다 |

| 규제 매개변수 | 뜻 | 규제를 세게 하려면 |
|--------------|-----|------------------|
| `max_depth` | 트리의 최대 깊이 (가장 직관적) | **줄인다** |
| `max_leaf_nodes` | 리프 노드의 최대 개수 | **줄인다** |
| `max_features` | 각 노드 분할 때 고려할 최대 특성 수 | **줄인다** |
| `min_samples_split` | 노드를 나누기 위한 최소 샘플 수 | **늘린다** |
| `min_samples_leaf` | 리프 노드가 가져야 할 최소 샘플 수 | **늘린다** |

→ **`max_` 는 줄이고, `min_` 은 늘리면** 규제가 세져 과대적합이 줄어든다.

![깊이에 따른 과대적합](./images/anim_02_depth_overfitting.gif)

In [ ]:
# max_depth 를 1 ~ 15 로 바꿔 가며 학습용 / 평가용 RMSE 비교
rows = []
for depth in range(1, 16):
    model = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({
        'max_depth': depth,
        '학습용 RMSE': round(rmse(y_train, model.predict(X_train)), 2),
        '평가용 RMSE': round(rmse(y_test, model.predict(X_test)), 2),
    })
result = pd.DataFrame(rows)

# 그래프
plt.figure(figsize=(9, 4))
plt.plot(result['max_depth'], result['학습용 RMSE'], marker='o', label='학습용 RMSE')
plt.plot(result['max_depth'], result['평가용 RMSE'], marker='o', label='평가용 RMSE')
plt.xlabel('max_depth')
plt.ylabel('RMSE')
plt.legend()
plt.show()

result.head(8)

In [ ]:
# min_samples_leaf : 리프에 최소 몇 채는 남기기
for leaf in [1, 5, 10, 20]:
    model = DecisionTreeRegressor(min_samples_leaf=leaf, random_state=0).fit(X_train, y_train)
    print(f"min_samples_leaf = {leaf:>2} → 깊이 {model.get_depth():>2}, "
          f"학습용 {rmse(y_train, model.predict(X_train)):.2f} / 평가용 {rmse(y_test, model.predict(X_test)):.2f}")

> - 깊이가 깊어질수록 **학습용 RMSE 는 0 으로** 계속 줄지만, **평가용은 4.9 ~ 5.8 사이에서 들쭉날쭉**하다 (깊이 6 에서 4.92 로 가장 낮음)
> - `min_samples_leaf` 를 20 으로 늘리면 깊이 5 에서 멈추고 평가용 **5.25** 로 조금 나아진다
> - 깊이 하나만 바꿔도 평가용 오차가 크게 흔들린다 → 바로 다음의 **불안정성**

## 4. 결정 트리의 불안정성 (Instability) (교재 p.90)

| 한계 | 내용 |
|------|------|
| **① 축 수직 분할의 한계** | 공간을 **축에 수직으로만** 자른다. 진짜 경계가 대각선 · 곡선이면 수많은 **계단**으로 흉내 내야 하고, 매끄럽지 못한 경계가 오차를 만든다 |
| **② 높은 분산 · 민감성** | 학습 데이터가 **조금만 바뀌어도**(샘플 추가 · 삭제) 트리 구조 전체가 완전히 달라질 수 있다 |

**해결 방안** — 여러 트리를 결합하는 **앙상블**(배깅, 부스팅) → [`random_forest.ipynb`](./random_forest.ipynb)

![트리 불안정성 시뮬레이션](./images/anim_03_instability.gif)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 진짜 경계가 '대각선(y = x)' 인 2차원 데이터 220개 (노이즈 조금)
rng = np.random.RandomState(1)
P = rng.uniform(0, 10, size=(220, 2))
label = (P[:, 1] > P[:, 0] + rng.normal(0, 0.9, size=220)).astype(int)

# 12개만 빼 본 데이터
keep = np.ones(len(P), dtype=bool)
keep[rng.choice(len(P), 12, replace=False)] = False

models = [
    ('단일 트리 (220개)', DecisionTreeClassifier(random_state=0).fit(P, label)),
    ('단일 트리 (12개 빼고)', DecisionTreeClassifier(random_state=0).fit(P[keep], label[keep])),
    ('랜덤 포레스트', RandomForestClassifier(random_state=0).fit(P, label)),
]

# 격자 위의 모든 점을 예측해서 경계 그리기
xx, yy = np.meshgrid(np.linspace(0, 10, 300), np.linspace(0, 10, 300))
grid2 = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, (name, model) in zip(axes, models):
    Z = model.predict(grid2).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.plot([0, 10], [0, 10], 'k--', lw=1)               # 진짜 경계
    ax.scatter(P[:, 0], P[:, 1], c=label, cmap='coolwarm', s=12)
    ax.set_title(name)
plt.show()

# 두 단일 트리의 루트 질문이 같은가?
for name, model in models[:2]:
    t = model.tree_
    print(f"{name} → 루트 : x{t.feature[0]} <= {t.threshold[0]:.2f}, 깊이 {model.get_depth()}")

> - 단일 트리의 경계는 대각선을 **계단**으로 흉내 낸다 (①)
> - 220개 중 **12개만 빼도** 루트 질문부터 `x1 <= 5.76` → `x0 <= 5.55` 로 바뀌고, 계단 모양이 눈에 띄게 달라진다 (②)
> - 랜덤 포레스트는 트리 100개의 다수결이라 **대각선에 더 가깝고**, 데이터 몇 개에 덜 흔들린다

### 보스턴 데이터로 확인 — 분할을 10번 바꿔 보기

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# train_test_split 의 random_state 를 0 ~ 9 로 바꿔 10번 다시 나누고, 매번 평가용 RMSE 측정
scores = []
for seed in range(10):
    Xa, Xb, ya, yb = train_test_split(X_data, y_data, test_size=0.25, random_state=seed)
    dt_s = DecisionTreeRegressor(random_state=0).fit(Xa, ya)
    rf_s = RandomForestRegressor(random_state=0).fit(Xa, ya)
    scores.append({'split': seed,
                   '결정 트리': rmse(yb, dt_s.predict(Xb)),
                   '랜덤 포레스트': rmse(yb, rf_s.predict(Xb))})

scores = pd.DataFrame(scores).set_index('split').round(2)
print(scores.T)
print("-" * 40)
print(scores.agg(['mean', 'std']).round(2))

> - 결정 트리 평가용 RMSE 는 평균 **5.19**, 표준편차 **0.86** — 데이터를 어떻게 나누느냐에 따라 크게 흔들린다
> - 랜덤 포레스트는 평균 **3.44**, 표준편차 **0.37** — 더 정확하고 더 **안정적**이다

## 5. 오늘 정리 (트리 회귀)

| 항목 | 내용 |
|------|------|
| 예측값 | 리프에 속한 학습 데이터의 **평균** → 계단 모양 |
| 분할 기준 | **MSE** 가 가장 작아지는 특성 · 임계값 |
| 보스턴 결과 | 평가용 RMSE **5.41** (학습용 0.00 → 과대적합) |
| 규제 | `max_` 줄이기 · `min_` 늘리기 |
| 불안정성 | 축 수직 계단 경계 + 데이터에 민감 → **앙상블**로 해결 |